# Segmentacion RFM de clientes

**Que responde este cuaderno**

1. Como se reparte la base de clientes entre segmentos.
2. Que segmentos concentran el ingreso.
3. Cuanto vale el grupo de alto valor que se esta enfriando.
4. Que accion comercial va primero.

**Datos**: `rfm_segmentation_final.csv` (4.338 clientes con Recency, Frequency, Monetary, scores 1-4 y segmento). Se descarga por URL al ejecutar.

**Como correrlo**: Runtime > Run all. Solo pandas y matplotlib.

In [ ]:
import io, pathlib, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/Nicolenki7/Online-Retail-Business-Analysis/main/'
pathlib.Path('outputs').mkdir(exist_ok=True)

with urllib.request.urlopen(BASE + 'rfm_segmentation_final.csv') as r:
    d = pd.read_csv(io.BytesIO(r.read()))

print('clientes:', len(d))
print(d[['Recency', 'Frequency', 'Monetary']].describe().round(1).to_string())
d.head(3)

## 1. Tamano de cada segmento

Antes de decidir una accion hay que saber a cuanta gente alcanza.

In [ ]:
conteo = d['Segment'].value_counts()
cuota = (conteo / len(d) * 100).round(1)
print(pd.DataFrame({'clientes': conteo, 'pct': cuota}).to_string())

print()
print(f'RESPUESTA: el segmento mas grande es {conteo.index[0]} con {conteo.iloc[0]} clientes ({cuota.iloc[0]} %).')

ax = cuota.sort_values().plot(kind='barh', figsize=(7, 3.5))
ax.set_xlabel('% de clientes')
ax.set_title('Clientes por segmento')
plt.tight_layout()
plt.savefig('outputs/segmentos.png', dpi=150)
plt.show()

## 2. Que segmentos traen la plata

El tamano no es lo mismo que el valor: comparamos cuota de clientes contra cuota de gasto.

In [ ]:
g = d.groupby('Segment').agg(
    clientes=('CustomerID', 'count'),
    gasto=('Monetary', 'sum'),
    gasto_medio=('Monetary', 'mean'),
    recencia_media=('Recency', 'mean'),
)
g['pct_clientes'] = (g['clientes'] / g['clientes'].sum() * 100).round(1)
g['pct_gasto'] = (g['gasto'] / g['gasto'].sum() * 100).round(1)
g['gasto_medio'] = g['gasto_medio'].round(0)
g['recencia_media'] = g['recencia_media'].round(0)
g = g.sort_values('gasto', ascending=False)

print(g[['clientes', 'pct_clientes', 'pct_gasto', 'gasto_medio', 'recencia_media']].to_string())
g.to_csv('outputs/resumen_segmentos.csv')

print()
print(f'RESPUESTA: {g.index[0]} concentra el {g.pct_gasto.iloc[0]} % del gasto con solo el {g.pct_clientes.iloc[0]} % de los clientes.')

comparativa = g[['pct_clientes', 'pct_gasto']]
ax = comparativa.plot(kind='bar', figsize=(8, 3.5), rot=20)
ax.set_ylabel('%')
ax.set_title('Cuota de clientes vs cuota de gasto')
plt.tight_layout()
plt.savefig('outputs/cuota_vs_gasto.png', dpi=150)
plt.show()

## 3. El alto valor que se enfria

`No se pueden perder` es el segmento de alto gasto historico con compras antiguas: el que mas plata se lleva si migra a la competencia.

In [ ]:
riesgo = d[d['Segment'] == 'No se pueden perder']
if len(riesgo):
    print('clientes:', len(riesgo))
    print('gasto total:', round(riesgo['Monetary'].sum()))
    print('gasto medio:', round(riesgo['Monetary'].mean()))
    print('recencia media (dias):', round(riesgo['Recency'].mean()))
    print()
    print(f"RESPUESTA: son {len(riesgo)} clientes con {round(riesgo['Monetary'].sum())} de gasto historico y {round(riesgo['Recency'].mean())} dias sin comprar en promedio.")
else:
    print('El CSV no trae filas de ese segmento con este nombre exacto:', sorted(d['Segment'].unique()))

## 4. Prioridad: donde se juega el ingreso

Cruzamos gasto con recencia: arriba a la derecha esta el cliente valioso que ya casi no aparece.

In [ ]:
ax = d.plot(kind='scatter', x='Recency', y='Monetary', s=8, alpha=0.35, figsize=(7, 4))
ax.set_xlabel('Recency (dias desde la ultima compra)')
ax.set_ylabel('Monetary (gasto historico)')
ax.set_yscale('log')
ax.set_title('Valor vs recencia: el riesgo esta a la derecha')
plt.tight_layout()
plt.savefig('outputs/valor_vs_recencia.png', dpi=150)
plt.show()

top10 = d.nlargest(int(len(d) * 0.10), 'Monetary')
print(f"RESPUESTA: el 10 % de clientes de mayor gasto concentra el {top10['Monetary'].sum() / d['Monetary'].sum() * 100:.1f} % del ingreso.")
print('Accion: reactivacion sobre alto valor con recencia alta antes que captacion, porque el cliente ya pago el costo de adquisicion.')

## Limites

- Los scores 1-4 vienen calculados por cuartiles del propio dataset: un cliente '4' lo es respecto de esta base, no en absoluto.
- `Monetary` es gasto historico acumulado, no margen ni rentabilidad.
- Sin fecha de alta no se puede hacer cohortes ni ciclo de vida.
- Los nombres de segmento son etiquetas de negocio, no clusters estadisticos.